# Notebook 03: Document Chunking Strategies

Real documents are long. A 20-page PDF won't fit in a single embedding — and even if it did, it would retrieve the whole PDF for every query, burying the relevant parts in noise.

**Chunking** is the process of splitting documents into smaller, retrievable pieces.

## Why Chunking Matters

| Too Small | Too Large |
|---|---|
| Precise retrieval | Imprecise retrieval |
| Missing context | Lots of context |
| More chunks to store | Fewer chunks to store |
| May exceed token limits | May exceed token limits |

Finding the right chunk size is one of the most important RAG tuning decisions.

## Chunking Strategies Covered
1. **Fixed-size chunking** — split every N characters
2. **Sentence-based chunking** — split at sentence boundaries
3. **Recursive chunking** — try paragraph → sentence → word
4. **Overlapping chunks** — add overlap to preserve boundary context
5. **Semantic chunking** — split when topic changes (embedding-based)

## Setup

In [ ]:
import re
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import chromadb
from chromadb.utils import embedding_functions

# Load our sample knowledge base
kb_path = Path("../data/knowledge_base.txt")
raw_text = kb_path.read_text()

print(f"Knowledge base loaded: {len(raw_text)} characters")
print(f"\nFirst 300 characters:")
print(raw_text[:300])

## Strategy 1: Fixed-Size Chunking

The simplest approach — split every N characters. Fast and predictable, but may cut sentences mid-way.

In [ ]:
def fixed_size_chunks(text: str, chunk_size: int = 500, overlap: int = 0) -> list[str]:
    """
    Split text into fixed-size chunks.
    
    Args:
        text: Input text
        chunk_size: Characters per chunk
        overlap: Characters to overlap between chunks
    """
    chunks = []
    step = chunk_size - overlap
    for start in range(0, len(text), step):
        chunk = text[start:start + chunk_size].strip()
        if chunk:
            chunks.append(chunk)
    return chunks


chunks_fixed = fixed_size_chunks(raw_text, chunk_size=400, overlap=0)
chunks_overlap = fixed_size_chunks(raw_text, chunk_size=400, overlap=80)

print(f"Fixed-size (no overlap):  {len(chunks_fixed)} chunks")
print(f"Fixed-size (80 overlap):  {len(chunks_overlap)} chunks")
print(f"\nExample chunk (fixed):")
print(repr(chunks_fixed[1]))

Notice the chunk might cut off in the middle of a sentence. This is fixed-size chunking's main weakness.

## Strategy 2: Sentence-Based Chunking

Split at sentence boundaries so each chunk contains complete thoughts.

In [ ]:
def sentence_chunks(text: str, sentences_per_chunk: int = 3, overlap_sentences: int = 1) -> list[str]:
    """
    Split text into chunks of N sentences with optional sentence overlap.
    """
    # Split on sentence-ending punctuation
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s.strip() for s in sentences if s.strip()]
    
    chunks = []
    step = sentences_per_chunk - overlap_sentences
    for i in range(0, len(sentences), step):
        chunk = " ".join(sentences[i:i + sentences_per_chunk])
        if chunk:
            chunks.append(chunk)
    return chunks


chunks_sentence = sentence_chunks(raw_text, sentences_per_chunk=3, overlap_sentences=1)

print(f"Sentence-based chunks: {len(chunks_sentence)}")
print(f"\nExample chunk:")
print(chunks_sentence[2])

## Strategy 3: Document-Level Chunking

Our knowledge base has a clear structure — each DOCUMENT: section is a natural unit. This is the best strategy when your source has clear delimiters.

In [ ]:
def document_chunks(text: str) -> list[dict]:
    """
    Split text at DOCUMENT: boundaries, preserving titles as metadata.
    """
    sections = text.strip().split('\n\n')
    chunks = []
    for section in sections:
        section = section.strip()
        if section.startswith('DOCUMENT:'):
            lines = section.split('\n', 1)
            title = lines[0].replace('DOCUMENT:', '').strip()
            content = lines[1].strip() if len(lines) > 1 else ""
            if content:
                chunks.append({"title": title, "content": content})
    return chunks


doc_chunks = document_chunks(raw_text)

print(f"Document chunks: {len(doc_chunks)}")
print()
for chunk in doc_chunks:
    print(f"  [{len(chunk['content'])} chars] {chunk['title']}")

## Strategy 4: Recursive Chunking

Try to split on the most semantic boundary first, falling back to smaller ones. This is what LangChain's `RecursiveCharacterTextSplitter` does.

In [ ]:
def recursive_chunks(text: str, chunk_size: int = 400, overlap: int = 50) -> list[str]:
    """
    Recursively split text, trying larger separators first.
    Separator priority: paragraph → sentence → word → character
    """
    separators = ["\n\n", ".\n", ". ", "! ", "? ", ", ", " ", ""]

    def split_with_separator(text, separators):
        if not separators or len(text) <= chunk_size:
            return [text] if text.strip() else []

        sep = separators[0]
        parts = text.split(sep) if sep else list(text)
        chunks = []
        current = ""

        for part in parts:
            candidate = (current + sep + part).strip() if current else part.strip()
            if len(candidate) <= chunk_size:
                current = candidate
            else:
                if current:
                    chunks.append(current)
                # If a single part is too large, recurse
                if len(part) > chunk_size:
                    chunks.extend(split_with_separator(part, separators[1:]))
                    current = ""
                else:
                    current = part.strip()

        if current:
            chunks.append(current)

        return chunks

    return split_with_separator(text, separators)


chunks_recursive = recursive_chunks(raw_text, chunk_size=400, overlap=50)

print(f"Recursive chunks: {len(chunks_recursive)}")
print(f"\nChunk sizes:")
sizes = [len(c) for c in chunks_recursive]
print(f"  Min: {min(sizes)} chars")
print(f"  Max: {max(sizes)} chars")
print(f"  Avg: {sum(sizes)/len(sizes):.0f} chars")

print(f"\nExample chunk:")
print(chunks_recursive[3])

## Comparing Strategies Side by Side

In [ ]:
strategies = {
    "Fixed-size (no overlap)": chunks_fixed,
    "Fixed-size (with overlap)": chunks_overlap,
    "Sentence-based": chunks_sentence,
    "Document-level": [c["content"] for c in doc_chunks],
    "Recursive": chunks_recursive,
}

print(f"{'Strategy':<30} {'# Chunks':>10} {'Avg Chars':>12} {'Min':>8} {'Max':>8}")
print("-" * 72)
for name, chunks in strategies.items():
    sizes = [len(c) for c in chunks]
    print(f"{name:<30} {len(chunks):>10} {sum(sizes)/len(sizes):>12.0f} {min(sizes):>8} {max(sizes):>8}")

## Storing Chunks in ChromaDB

Let's store our document-level chunks (the best strategy for this knowledge base) in ChromaDB.

In [ ]:
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

client = chromadb.PersistentClient(path="../data/chroma_rag")

try:
    client.delete_collection("knowledge_base")
except:
    pass

collection = client.create_collection(
    name="knowledge_base",
    embedding_function=embedding_fn
)

# Add all document chunks
collection.add(
    documents=[chunk["content"] for chunk in doc_chunks],
    ids=[f"doc_{i}" for i in range(len(doc_chunks))],
    metadatas=[{"title": chunk["title"]} for chunk in doc_chunks]
)

print(f"Stored {collection.count()} chunks in ChromaDB")

In [ ]:
# Test retrieval
results = collection.query(
    query_texts=["What is the difference between fine-tuning and RAG?"],
    n_results=3
)

print("Query: 'What is the difference between fine-tuning and RAG?'")
print("-" * 60)
for doc, meta, dist in zip(
    results['documents'][0],
    results['metadatas'][0],
    results['distances'][0]
):
    print(f"\n  Title: {meta['title']}")
    print(f"  Score: {1/(1+dist):.3f}")
    print(f"  Text: {doc[:200]}...")

## Key Takeaways

1. **Chunking strategy significantly impacts retrieval quality** — there's no single best approach
2. **Fixed-size** is simple but breaks sentences; good for a quick start
3. **Sentence/document-level** preserves semantic units; better for most use cases
4. **Overlap** helps avoid losing context at chunk boundaries
5. **Use natural structure** in your documents (headers, paragraphs) whenever possible

**Rule of thumb:** Start with document-level or paragraph-level chunks, then tune based on retrieval quality.

---

**Next:** Notebook 04 — The full end-to-end RAG pipeline, combining retrieval with LLM generation.